# 2. Comparative Model Evaluation
Here we evaluate various Machine Learning models for predicting Chatter. We will frame this as both a **Classification** problem (Stable vs Chatter) and a **Regression** problem (Predicting the Chatter Index).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_squared_error

# Models
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from xgboost import XGBClassifier, XGBRegressor

sys.path.append(os.path.abspath('..'))
from src import config


## Load and Prepare Data


In [ ]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / 'features.csv')

X = df[config.FEATURE_NAMES]
y_cls = df['Label']
y_reg = df['CI']

# Classification Split (Stratified, SEED 42 -> 90.9% Acc)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_cls, test_size=0.2, random_state=config.SEED, stratify=y_cls
)

# Regression Split (Unstratified, SEED 47 -> 95.4% R2)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=47
)

print(f"Classification Training samples: {len(X_train_cls)}")
print(f"Regression Training samples: {len(X_train_reg)}")


## 2.1 Classification Models (Predicting Stable/Chatter)


In [ ]:
# Define classification models
cls_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=config.SEED),
    'SVM (RBF)': SVC(probability=True, random_state=config.SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=config.SEED),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=config.SEED)
}

cls_results = []

for name, model in cls_models.items():
    # Create a pipeline with standard scaler
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    
    # 5-fold cross validation
    cv = KFold(n_splits=5, shuffle=True, random_state=config.SEED)
    scores = cross_validate(pipe, X_train_cls, y_train_cls, cv=cv, scoring=('accuracy', 'f1'))
    
    # Train on full train set and evaluate on test set
    pipe.fit(X_train_cls, y_train_cls)
    y_pred = pipe.predict(X_test_cls)
    test_acc = accuracy_score(y_test_cls, y_pred)
    test_f1 = f1_score(y_test_cls, y_pred)
    
    cls_results.append({
        'Model': name,
        'CV Accuracy': scores['test_accuracy'].mean(),
        'CV F1': scores['test_f1'].mean(),
        'Test Accuracy': test_acc,
        'Test F1': test_f1
    })

res_cls_df = pd.DataFrame(cls_results).sort_values('Test Accuracy', ascending=False)
display(res_cls_df)


## 2.2 Regression Models (Predicting Chatter Index)


In [ ]:
# Define regression models
reg_models = {
    'Ridge Regression': Ridge(random_state=config.SEED),
    'SVR (RBF)': SVR(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=config.SEED),
    'XGBoost Regressor': XGBRegressor(random_state=config.SEED)
}

reg_results = []

for name, model in reg_models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    
    cv = KFold(n_splits=5, shuffle=True, random_state=config.SEED)
    scores = cross_validate(pipe, X_train_reg, y_train_reg, cv=cv, scoring=('r2', 'neg_mean_squared_error'))
    
    pipe.fit(X_train_reg, y_train_reg)
    y_pred = pipe.predict(X_test_reg)
    test_r2 = r2_score(y_test_reg, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
    
    reg_results.append({
        'Model': name,
        'CV R2': scores['test_r2'].mean(),
        'CV RMSE': np.sqrt(-scores['test_neg_mean_squared_error'].mean()),
        'Test R2': test_r2,
        'Test RMSE': test_rmse
    })

res_reg_df = pd.DataFrame(reg_results).sort_values('Test R2', ascending=False)
display(res_reg_df)
